# Exploratory Analysis for NOAA Weather and Climate Dataset

[![Open in SageMaker Studio Lab](https://studiolab.sagemaker.aws/studiolab.svg)](https://studiolab.sagemaker.aws/import/github/aws/studio-lab-examples/blob/main/geospatial-data-science/NOAA_Exploratory_Analysis/EDA_weather_climate.ipynb)

In this notebook, we go through basic steps of exploratory data analysis (EDA), performing initial data investigations to discover patterns, spot anomalies, and look for insights to inform later ML modeling choices.

We will use some daily weather data from a NOAA (National Oceanic and Atmospheric Administration) station in Asheville, NC. A full description of this dataset is available at:

https://www.ncei.noaa.gov/pub/data/uscrn/products/daily01/

In [ ]:
# Importing libraries
import numpy as np 
import pandas as pd 
import matplotlib.pyplot as plt
import scipy.stats
import seaborn as sns
import datetime
import zipfile
import importlib
import subprocess
import sys
import os
from pylab import rcParams
import warnings
warnings.filterwarnings('ignore')

In [ ]:
# Prerequisites panda update to 1.0.5 version
if pd.__version__ < '1.0.5':
    subprocess.check_call([sys.executable, '-m', 'conda', 'install', '-y', 'pandas==1.0.5'])
    importlib.reload(pd)

## Downloading the dataset

In [ ]:
data_base = 'data/daily01/'
url_base = 'https://www.ncei.noaa.gov/pub/data/uscrn/products/daily01/snapshots/'
dataset_zip = 'CRND0103-202110250450.zip'
if not os.path.isdir(data_base):
    !mkdir -p {data_base}
    
# Download dataset from NOAA website
if not os.path.isfile(data_base+dataset_zip):
    !wget --no-check-certificate {url_base}{dataset_zip}
    !mv {dataset_zip} {data_base}

# Uncompress the dataset
if not os.path.isfile(data_base+'HEADERS.txt'):
    print("Uncompressing the dataset")
    with zipfile.ZipFile(data_base+dataset_zip, 'r') as zip_ref:
        zip_ref.extractall(data_base)    

## Overall Statistics

### Variables list

In [ ]:
with open(data_base + "HEADERS.txt") as file:  
    data = file.read()
lines = data.split('\n')   
headers = lines[1].split(' ')[:-1]
print(headers)

### Reading three years of data from Asheville, NC

In [ ]:
dframes = []
for year in range(2017, 2020):
    data_file = f'{year}/CRND0103-{year}-NC_Asheville_13_S.txt'               
    df = pd.read_csv(data_base + data_file, parse_dates=[1],
                     names=headers, header=None, sep='\s+',
                     na_values=[-9999.0, -99.0])
    dframes.append(df)

df = pd.concat(dframes, ignore_index=True)

In [ ]:
df.head()

Check for data types and non-null values for each column

In [ ]:
df.info()

In [ ]:
print(df.shape)

### Categorical variables

In [ ]:
categorical = df.select_dtypes(include = ["object"]).keys()
print(categorical)

Type of infrared surface temperature measurement (SUR_TEMP_DAILY_TYPE)

In [ ]:
df['SUR_TEMP_DAILY_TYPE'].value_counts()

### Numerical variables

In [ ]:
# Quantitative variables:
quantitative = df.select_dtypes(include = ["int64","float64"]).keys()
# Drop station WBAN number and version number of the station datalogger CRX_VN
quantitative = quantitative.drop(['WBANNO', 'CRX_VN'])
print(quantitative)

In [ ]:
df[quantitative].describe()

### Histograms for numerical

In [ ]:
rcParams['figure.figsize'] = 16, 18
df[quantitative].hist()

Look more closely into P_DAILY_CALC - Total amount of precipitation (mm)

In [ ]:
df['P_DAILY_CALC'].value_counts()

In [ ]:
df[df['P_DAILY_CALC']>0.0]['P_DAILY_CALC'].describe()

Majority on the days contains no rain, therefore we could add new features related to wet & dry days.

Histogram for rainy days

In [ ]:
fig = plt.figure(figsize = (14,7))
ax = fig.gca()
df[df['P_DAILY_CALC']>0.0]['P_DAILY_CALC'].hist(ax = ax)

## Data transformation -  Feature engineering

### Add new features for year, month, day

In [ ]:
# Date transformation
df['year'] = df['LST_DATE'].dt.year
df['month'] = df['LST_DATE'].dt.month
df['day'] = df['LST_DATE'].dt.day
df['dayofweek'] = df['LST_DATE'].dt.weekday

In [ ]:
df.head()

### Add categorical new feature for rainy days

#### Add rainy (Wet/Dry) new features

In [ ]:
# Categorical rainy feature
df['rainy'] = 'Wet'
df.loc[df['P_DAILY_CALC']==0.0,'rainy'] = 'Dry'
df['Dry'] = 0
df.loc[df['P_DAILY_CALC']==0.0,'Dry'] = 1
df['Wet'] = 0
df.loc[df['P_DAILY_CALC']!=0.0,'Wet'] = 1

In [ ]:
df['rainy'].value_counts()

#### Add rain label new feature using bins

In [ ]:
bins = [0, 1, 10, 20, 40, 110]
labels = ['No rain', 'Drizzle', 'Light rain', 'Moderate rain', 'Heavy rain']
df['rain_label'] = pd.cut(df.P_DAILY_CALC, bins, labels=labels, include_lowest=True)

In [ ]:
df['rain_label'].value_counts()

In [ ]:
rcParams['figure.figsize'] = 14, 7
sns.countplot(y=df['rain_label'])

Comparing Solar energy(radiance) with rain type

In [ ]:
sns.boxplot(x=df['rain_label'], y=df['SOLARAD_DAILY']).set(
    xlabel='Rain intensity', 
    ylabel='Solar energy',
    title='Comparing Solar energy(radiance) with rain type'
)

Comparing Solar energy(radiance) with Average air temperature

In [ ]:
jointplot = sns.jointplot(data=df, x="T_DAILY_AVG", y="SOLARAD_DAILY", kind="reg", height=10)
jointplot.set_axis_labels('Air temperature', 'Solar energy', fontsize=16)

We can notice little correlation between these two variables, lets have a look to overall correlations between main variable in the next section.

## Data Exploration

### Overall Correlations

In [ ]:
correlations_vars=['rainy','T_DAILY_AVG', 'P_DAILY_CALC', 'SOLARAD_DAILY', 'SUR_TEMP_DAILY_AVG', 'RH_DAILY_AVG', 
              'SOIL_MOISTURE_5_DAILY', 'SOIL_TEMP_5_DAILY']
df[correlations_vars].head()

#### Correlation Matrix

In [ ]:
sns.pairplot(df[correlations_vars], hue='rainy')

A couple of observations from the correlation matrix:
- strong correlation between air temperature and soil temperature and also with infrared surface temperature
- little correlation between solar energy and humidity
- little correlation between soil moisture and infrared surface temperature
- air humidity is not correlated with air temperature 


### Visualizations by Date

#### Index by date

In [ ]:
df_idx = df.set_index('LST_DATE')
df_idx.head()

#### Showing number of Dry/Wet days per month

In [ ]:
rcParams['figure.figsize'] = 16, 8
df_idx.groupby(df_idx.index.month).aggregate({'Dry': 'sum', 'Wet': 'sum'}).plot(kind='bar', title='Number of Dry/Wet days per month')
plt.xlabel('Month')
plt.ylabel('Number of days')

#### Average air temperature by date

In [ ]:
df_idx.T_DAILY_AVG.plot(title='Average air temperature')
plt.xlabel('Date')
plt.ylabel('Air temperature')

We can notice sesonability, air temperature varies with the seasons

### HeatMaps

In [ ]:
# Drawing a heatmap
def weather_heatmap(data, color, **kws):
    values=data.columns.values[3]
    data = data.pivot(index='month', columns='day', values=values)
    sns.heatmap(data, cmap='coolwarm', **kws)  

# Joining heatmaps of every month in a year 
def weather_calendar(weather): 
    dfyear = df[['year', 'month', 'day', weather]]
    vmin=dfyear[weather].min()
    vmax=dfyear[weather].max()
    with sns.plotting_context(font_scale=12):
        g = sns.FacetGrid(dfyear,col="year", col_wrap=3) #One heatmap per month
        g = g.map_dataframe(weather_heatmap,vmin=vmin, vmax=vmax)
        g.set_axis_labels('Day', 'Month')
        plt.subplots_adjust(top=0.8)
        g.fig.suptitle('%s Calendar' %(weather))

#### Temperature heatmap

In [ ]:
weather_calendar('T_DAILY_AVG')

As expected the air temperature has higher values during the summer months. 

#### Precipitation heatmap

In [ ]:
weather_calendar('P_DAILY_CALC')

The climate is not very rainy in Asheville.

#### Humidity heatmap

In [ ]:
weather_calendar('RH_DAILY_AVG')

Even if it doesn't rain heavily, there are a lot of days with important humidity. 